# 02 — Single-cell (GSE132465)

QC, normalize, HVGs, **patient subset** (merge two patients if one has <2000 cells after QC), Leiden + marker-based cell type labels.

**Data source:** GEO supplementary archive for GSE132465. First run downloads a large `.tar` — ensure enough disk space.


In [1]:
USE_DRIVE = False
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BulkCellGNN_data")
else:
    DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR.resolve())


DATA_DIR = C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data


In [2]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scanpy", "anndata", "numpy", "pandas", "scipy", "matplotlib", "scikit-learn",
    "python-igraph", "leidenalg", "GEOparse"])


0

In [3]:
import os, random, json, tarfile, urllib.request
SEED = 42
random.seed(SEED)
os.environ.setdefault("PYTHONHASHSEED", str(SEED))
import numpy as np
np.random.seed(SEED)

import scanpy as sc
sc.settings.verbosity = 2
try:
    sc.settings.seed = int(SEED)
except Exception:
    pass

import pandas as pd


In [5]:
# GSE132465 stores data as a single processed count matrix, not a RAW.tar.gz
# Correct file: GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt.gz

import urllib.request
from pathlib import Path

MATRIX_FILE = DATA_DIR / "GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt.gz"
GEO_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132465/suppl/"
    "GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt.gz"
)

if not MATRIX_FILE.exists():
    print(f"Downloading {GEO_URL} ...")
    print("This file is ~1.5 GB compressed — will take several minutes.")
    urllib.request.urlretrieve(GEO_URL, MATRIX_FILE)
    print("Download complete.")
else:
    print("Found existing:", MATRIX_FILE)

print("File size:", round(MATRIX_FILE.stat().st_size / 1e9, 2), "GB")


This file is ~1.5 GB compressed — will take several minutes.
Download complete.
File size: 0.13 GB


In [6]:
# File structure verification
import gzip

with gzip.open(MATRIX_FILE, 'rt') as f:
    header = f.readline().strip()
    row2   = f.readline().strip()

print("Header (first 300 chars):")
print(header[:300])
print("\nRow 2 (first 300 chars):")
print(row2[:300])
print("\nNumber of columns (cells) in header:", len(header.split('\t')))

Header (first 300 chars):
Index	SMC01-T_AAACCTGCATACGCCG	SMC01-T_AAACCTGGTCGCATAT	SMC01-T_AAACCTGTCCCTTGCA	SMC01-T_AAACGGGAGGGAAACA	SMC01-T_AAACGGGGTATAGGTA	SMC01-T_AAAGATGAGGCCGAAT	SMC01-T_AAAGATGCATGGATGG	SMC01-T_AAAGATGTCACGACTA	SMC01-T_AAAGATGTCCGTTGCT	SMC01-T_AAAGCAACAGTCGATT	SMC01-T_AAAGTAGAGAGGTACC	SMC01-T_AAAGTAGAGGG

Row 2 (first 300 chars):
A1BG	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0

Number of columns (cells) in header: 63690


In [7]:
import pandas as pd
import gc

# This file is genes x cells — we read it and immediately subset
# Reading full file needs ~8GB RAM — we chunk to stay within 16GB laptop limit

PATIENT_PREFIX = "SMC01"   # change if SMC01 has < 2000 cells after QC

print(f"Loading matrix, subsetting to patient {PATIENT_PREFIX}...")
print("Reading header to find SMC01 cell columns...")

# Step 1: read just the header to find which columns belong to SMC01
with gzip.open(MATRIX_FILE, 'rt') as f:
    header_line = f.readline().strip()

all_barcodes = header_line.split('\t')          # first element may be empty or "gene"
print(f"Total barcodes in file: {len(all_barcodes)}")
print(f"First 5: {all_barcodes[:5]}")
print(f"Sample barcode format: {all_barcodes[1] if len(all_barcodes) > 1 else 'N/A'}")

# Find columns belonging to our patient
patient_cols = [i for i, b in enumerate(all_barcodes) 
                if PATIENT_PREFIX.upper() in b.upper()]
print(f"\nColumns for {PATIENT_PREFIX}: {len(patient_cols)}")

if len(patient_cols) < 500:
    print(f"WARNING: only {len(patient_cols)} cells found for {PATIENT_PREFIX}.")
    print("Consider trying SMC02, SMC03, or merging two patients.")
    print("Available patient prefixes (sample):", 
          list({b.split('_')[0] for b in all_barcodes[:2000] if '_' in b})[:15])

Loading matrix, subsetting to patient SMC01...
Reading header to find SMC01 cell columns...
Total barcodes in file: 63690
First 5: ['Index', 'SMC01-T_AAACCTGCATACGCCG', 'SMC01-T_AAACCTGGTCGCATAT', 'SMC01-T_AAACCTGTCCCTTGCA', 'SMC01-T_AAACGGGAGGGAAACA']
Sample barcode format: SMC01-T_AAACCTGCATACGCCG

Columns for SMC01: 5226


In [12]:
# Read the full matrix but keep only patient columns
# usecols saves memory by not loading 63k columns at once

gene_col_idx = 0   # first column is gene name — adjust if peek shows differently

cols_to_load = [gene_col_idx] + patient_cols

print(f"Reading {len(cols_to_load)-1} cell columns from compressed matrix...")
print("This takes 3-8 minutes on a laptop...")

df = pd.read_csv(
    MATRIX_FILE,
    sep='\t',
    index_col=0,         # genes as index
    usecols=cols_to_load,
    compression='gzip',
)

print(f"Loaded shape (genes x cells): {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


Reading 5226 cell columns from compressed matrix...
This takes 3-8 minutes on a laptop...
Loaded shape (genes x cells): (33694, 5226)
Memory usage: 1.41 GB


In [13]:
# QC
import anndata as ad
import scanpy as sc

# Transpose: AnnData wants cells x genes
X = df.values.T.astype('float32')
cell_barcodes = df.columns.tolist()
gene_names    = df.index.tolist()

adata = ad.AnnData(
    X=X,
    obs=pd.DataFrame(index=cell_barcodes),
    var=pd.DataFrame(index=gene_names),
)
del df; gc.collect()   # free the pandas DataFrame immediately

print("AnnData created:", adata)

# Standard QC
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

mt_col = next((c for c in adata.obs.columns 
               if c.startswith("pct_counts_") and "mt" in c.lower()), None)
if mt_col:
    adata = adata[adata.obs[mt_col] < 20, :].copy()

print(f"After QC: {adata.shape}")

# Fallback: if fewer than 2000 cells, warn — add SMC02 manually
if adata.n_obs < 2000:
    print(f"WARNING: only {adata.n_obs} cells after QC.")
    print("Re-run the loading cells with PATIENT_PREFIX = 'SMC02' and concatenate.")


AnnData created: AnnData object with n_obs × n_vars = 5226 × 33694
filtered out 14149 genes that are detected in less than 3 cells
After QC: (5226, 19545)


In [16]:
# PCA / neighbors / UMAP / Leiden (reproducible)
sc.tl.pca(adata, svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(adata, resolution=0.5, random_state=SEED, 
             flavor="igraph", n_iterations=2, directed=False)

# Marker-based coarse cell types (replace with your manual annotations if needed)
markers = {
    "T_cell": ["CD3D", "CD3E"],
    "B_cell": ["CD19", "MS4A1"],
    "Myeloid": ["LYZ", "CD68"],
    "Epithelial": ["EPCAM", "KRT8"],
    "Stromal": ["DCN", "COL1A1"],
    "NK": ["NKG7", "GNLY"],
}
present = {k: [g for g in genes if g in adata.var_names] for k, genes in markers.items()}

def label_cluster(cluster_id: str) -> str:
    sub = adata[adata.obs["leiden"] == cluster_id]
    scores = {}
    for ct, genes in present.items():
        if not genes:
            scores[ct] = 0.0
            continue
        X_sub = sub[:, genes].X
        if hasattr(X_sub, "toarray"):
            X_sub = X_sub.toarray()
        scores[ct] = float(np.asarray(X_sub, dtype=np.float32).mean())
    return max(scores, key=scores.get)

clusters = sorted(adata.obs["leiden"].unique(), key=lambda x: (len(x), x))
map_leiden_to_type = {c: label_cluster(c) for c in clusters}
adata.obs["cell_type_str"] = adata.obs["leiden"].map(map_leiden_to_type)

type_names = sorted(adata.obs["cell_type_str"].unique())
type_to_idx = {n: i for i, n in enumerate(type_names)}
adata.obs["cell_type"] = adata.obs["cell_type_str"].map(type_to_idx).astype(int)

print(type_names)


computing PCA
    with n_comps=50
    finished (0:00:04)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished (0:00:00)
computing UMAP
    finished (0:00:13)
running Leiden clustering
    finished (0:00:00)
['Epithelial', 'Myeloid', 'NK', 'Stromal', 'T_cell']


In [17]:
# Package outputs for graph + model notebooks (HVG expression)
X = adata.X
if hasattr(X, "toarray"):
    cell_expr = X.toarray().astype(np.float32)
else:
    cell_expr = np.asarray(X, dtype=np.float32)

cell_genes = np.asarray(adata.var_names.astype(str))
cell_types = adata.obs["cell_type"].to_numpy(dtype=np.int64)
cell_umap = adata.obsm["X_umap"].astype(np.float32)

np.save(DATA_DIR / "cell_expr.npy", cell_expr)
np.save(DATA_DIR / "cell_genes.npy", cell_genes)
np.save(DATA_DIR / "cell_types.npy", cell_types)
np.save(DATA_DIR / "cell_umap.npy", cell_umap)
(DATA_DIR / "cell_type_names.json").write_text(json.dumps(type_names), encoding="utf-8")
print("Saved cell arrays + cell_type_names.json")


Saved cell arrays + cell_type_names.json


#### Notebook 02 — Summary and Results

**Dataset:** GSE132465 — Single-cell 3′ RNA sequencing of 23 Korean CRC patients
(Lee et al., 2020, *Nature Genetics*). Total dataset: 63,689 cells across 23 patients.

**Patient subset used:** SMC01 (tumor sample, suffix `-T`).

**Pipeline steps completed:**

| Step | Input | Output |
|---|---|---|
| Download | GEO FTP (0.13 GB compressed) | `GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt.gz` |
| Patient subset | 63,690 barcodes | 5,226 SMC01-T barcodes |
| Load | 33,694 genes × 5,226 cells | `df` (1.41 GB in memory) |
| QC — filter genes | min 3 cells | 33,694 → 19,545 genes retained |
| QC — filter cells | min 200 genes, MT% < 20 | 5,226 cells retained (0 lost) |
| Normalize + log1p | raw UMI counts | log-normalised to 10,000 per cell |
| HVG selection | 19,545 genes | 3,000 highly variable genes |
| PCA | 3,000 HVGs | 50 components (4 sec) |
| Leiden clustering | resolution = 0.5 | clusters → 5 cell types |

**Cell types identified (marker-based annotation):**

| Label | Index | Marker genes used | Biological role |
|---|---|---|---|
| Epithelial | 0 | EPCAM, KRT8 | Tumor cells |
| Myeloid | 1 | LYZ, CD68 | Macrophages / monocytes |
| NK | 2 | NKG7, GNLY | Innate cytotoxic immune cells |
| Stromal | 3 | DCN, COL1A1 | Cancer-associated fibroblasts |
| T_cell | 4 | CD3D, CD3E | Adaptive immune cells |

B_cell not detected as a distinct cluster at resolution 0.5 — minor population
likely merged into T_cell or Myeloid cluster in this single patient.

**Quality notes:**
- 0 cells lost to MT% filtering → SMC01 is a clean, high-quality sample
- 14,149 genes removed as lowly expressed (< 3 cells) — expected for scRNA-seq dropout
- No cell merging needed (5,226 > 2,000 cell minimum threshold)

**Files saved to `DATA_DIR`:**
- `cell_expr.npy` — (5226, 3000) float32 HVG expression matrix
- `cell_genes.npy` — (3000,) HVG gene names
- `cell_types.npy` — (5226,) integer cell type labels (0–4)
- `cell_umap.npy` — (5226, 2) UMAP coordinates for visualisation
- `cell_type_names.json` — `["Epithelial","Myeloid","NK","Stromal","T_cell"]`

**Relevance to BulkCell-GNN:**
The five cell types above become the categories in the C→B cell-type-aware
pooling (Rule 3). After training, the γₜ attention heatmap will show which of
these five types each bulk MSI/MSS sample attends to most strongly.
Expected biological finding: MSI samples → high T_cell / NK attention
(immune-hot TME); MSS samples → high Epithelial / Stromal attention
(immune-cold TME). This is the primary interpretability output of the project.

**Next:** `03_graph_construction.ipynb` builds the three edge types
(B–B, C–C, B–C) connecting bulk samples and single cells.